In [1]:
import sys
import json
import os
import numpy as np
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.pipeline import Pipeline

# Allow imports from src/
sys.path.insert(0, os.path.join(os.getcwd(), ".."))
from src.preprocessing import LABEL_NAMES, binarize_labels
from src.metrics import compute_metrics

In [2]:
# 2. Load GoEmotions Dataset
dataset = load_dataset("go_emotions", "simplified")

train_ds = dataset["train"]
val_ds   = dataset["validation"]
test_ds  = dataset["test"]

print(f"train: {len(train_ds):,}  val: {len(val_ds):,}  test: {len(test_ds):,}")

README.md: 0.00B [00:00, ?B/s]

simplified/train-00000-of-00001.parquet:   0%|          | 0.00/2.77M [00:00<?, ?B/s]

simplified/validation-00000-of-00001.par(…):   0%|          | 0.00/350k [00:00<?, ?B/s]

simplified/test-00000-of-00001.parquet:   0%|          | 0.00/347k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

train: 43,410  val: 5,426  test: 5,427


In [3]:
# 3. Build Feature Matrices
def build_xy(split):
    texts  = split["text"]
    labels = np.vstack([binarize_labels(row) for row in split["labels"]])
    return texts, labels

X_train, y_train = build_xy(train_ds)
X_val,   y_val   = build_xy(val_ds)
X_test,  y_test  = build_xy(test_ds)

print(f"X_train: {len(X_train):,} samples  y_train shape: {y_train.shape}")

X_train: 43,410 samples  y_train shape: (43410, 28)


In [4]:
# 4. Train Pipeline
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=50_000, sublinear_tf=True)),
    ("clf",   MultiOutputClassifier(
                  LogisticRegression(max_iter=1000, solver="lbfgs", C=1.0),
                  n_jobs=-1
              )),
])

pipeline.fit(X_train, y_train)
print("Training complete.")

Training complete.


In [5]:
# 5. Threshold Sweep on Validation Set
X_val_tfidf = pipeline["tfidf"].transform(X_val)
val_proba = np.column_stack(
    [est.predict_proba(X_val_tfidf)[:, 1] for est in pipeline["clf"].estimators_]
)

thresholds = [0.3, 0.4, 0.5]
best_threshold, best_micro_f1 = None, -1.0

print(f"{'threshold':>10}  {'micro_f1':>10}  {'macro_f1':>10}  {'precision':>10}  {'recall':>10}")
print("-" * 58)

for t in thresholds:
    preds = (val_proba >= t).astype(int)
    m = compute_metrics(y_val, preds)
    print(f"{t:>10.1f}  {m['micro_f1']:>10.4f}  {m['macro_f1']:>10.4f}"
          f"  {m['precision']:>10.4f}  {m['recall']:>10.4f}")
    if m["micro_f1"] > best_micro_f1:
        best_micro_f1  = m["micro_f1"]
        best_threshold = t

print(f"\nBest threshold: {best_threshold}  (micro F1 = {best_micro_f1:.4f})")

 threshold    micro_f1    macro_f1   precision      recall
----------------------------------------------------------
       0.3      0.5257      0.3315      0.5942      0.4713
       0.4      0.4868      0.2864      0.6574      0.3865
       0.5      0.4153      0.2439      0.7114      0.2933

Best threshold: 0.3  (micro F1 = 0.5257)


In [6]:
# 6. Evaluate on Test Set
X_test_tfidf = pipeline["tfidf"].transform(X_test)
test_proba = np.column_stack(
    [est.predict_proba(X_test_tfidf)[:, 1] for est in pipeline["clf"].estimators_]
)
test_preds   = (test_proba >= best_threshold).astype(int)
test_metrics = compute_metrics(y_test, test_preds)

print(f"Test set results (threshold = {best_threshold}):")
for k, v in test_metrics.items():
    print(f"  {k}: {v}")

Test set results (threshold = 0.3):
  micro_f1: 0.5287
  macro_f1: 0.3268
  precision: 0.5899
  recall: 0.4791


In [7]:
# 7. Save Predictions to `outputs/baseline_preds.jsonl
output_path = os.path.normpath(os.path.join(os.getcwd(), "..", "outputs", "baseline_preds.jsonl"))
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    for i, example in enumerate(test_ds):
        record = {
            "id":          example.get("id", i),
            "text":        example["text"],
            "true_labels": [LABEL_NAMES[j] for j, v in enumerate(y_test[i])  if v == 1],
            "pred_labels": [LABEL_NAMES[j] for j, v in enumerate(test_preds[i]) if v == 1],
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Saved {len(test_ds):,} predictions to: {output_path}")

Saved 5,427 predictions to: /Users/rongnanhe/info7160-goemotions-emotion-classification/outputs/baseline_preds.jsonl
